In [ ]:
import cv2
cap = cv2.VideoCapture('/nethome/agao81/flash/EgoVerse/external/scale/scripts/scale_data/695301bc5191b0ab09374f86/left_rectified.mp4')
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
cap.release()
print(f'Width: {width}, Height: {height}')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

parquet_path = Path("/nethome/agao81/flash/EgoVerse/external/scale/scripts/dataset/data/chunk-000/episode_000001.parquet")  # adjust

# Known shapes
IMG_SHAPE = (3, 1200, 1920)   # (C, H, W); update if different
ACTION_CHUNK = (100, 12)
KP_CHUNK = (100, 126)

def arr_from_bytes(b, shape, dtype):
    arr = np.frombuffer(b, dtype=dtype)
    return arr.reshape(shape)

df = pd.read_parquet(parquet_path)

# Pick a row to visualize
row = df.iloc[0]
# Decode columns
img_chw = arr_from_bytes(row["observations.images.front_img_1"], IMG_SHAPE, np.uint8)
img_hwc = np.transpose(img_chw, (1, 2, 0))

ee_chunk = arr_from_bytes(row["actions_ee_cartesian_cam"], ACTION_CHUNK, np.float32)
kp_chunk = arr_from_bytes(row["actions_ee_keypoints_world"], KP_CHUNK, np.float32)
ee_pose_now = arr_from_bytes(row["observations.state.ee_pose_cam"], (12,), np.float32)

print("Image shape:", img_hwc.shape)
print("EE chunk shape:", ee_chunk.shape)
print("Keypoints chunk shape:", kp_chunk.shape)
print("Current pose:", ee_pose_now)

# Simple trajectory overlay: plot right-hand xyz in image space (requires intrinsics)
# If you have intrinsics (fx, fy, cx, cy) from meta/info.json, use them here.
fx = 636.723
fy = 635.802
cx = 935.544
cy = 625.677
# Take the camera-frame xyz for right hand over the 100-step chunk
right_xyz_cam = ee_chunk[:, 6:9]  # (100, 3)

def project(points):
    z = np.clip(points[:, 2], 1e-3, None)
    u = fx * points[:, 0] / z + cx
    v = fy * points[:, 1] / z + cy
    return np.stack([u, v], axis=1)

pixels = project(right_xyz_cam)

plt.figure(figsize=(8, 5))
plt.imshow(img_hwc)
plt.scatter(pixels[:, 0], pixels[:, 1], c=np.linspace(0, 1, len(pixels)), cmap="viridis", s=12)
plt.title("Right-hand trajectory (chunk of 100)")
plt.axis("off")
plt.tight_layout()
plt.show()


# Palm pose points from camera-frame current pose (assumes [Lxyz, Lrot, Rxyz, Rrot])
left_palm_cam = ee_pose_now[0:3]
right_palm_cam = ee_pose_now[6:9]
palm_pix = project(np.vstack([left_palm_cam, right_palm_cam]))

# Overlay palms
plt.scatter(palm_pix[:, 0], palm_pix[:, 1], c=["cyan", "magenta"], s=40, marker="x")
plt.text(palm_pix[0, 0]+5, palm_pix[0, 1]-5, "L palm", color="cyan", fontsize=8)
plt.text(palm_pix[1, 0]+5, palm_pix[1, 1]-5, "R palm", color="magenta", fontsize=8)


# Optional: visualize keypoints in world frame (first timestep)
kp_world = kp_chunk[0].reshape(2, 21, 3)  # (hand, kp, xyz)
print("First frame right-hand world keypoints:", kp_world[1])

In [ ]:
import cv2
import os

# Write a short video with the right-hand trajectory overlaid on each frame
output_path = Path("trajectory_overlay.mp4")
fps = 10
max_frames = min(50, len(df))  # cap to dataset length

width = int(IMG_SHAPE[2])
height = int(IMG_SHAPE[1])

# Try multiple codecs until one opens
codecs = ["mp4v", "avc1", "H264", "XVID"]
writer = None
chosen = None
for c in codecs:
    fourcc = cv2.VideoWriter_fourcc(*c)
    w = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
    if w.isOpened():
        writer = w
        chosen = c
        break
    w.release()

if writer is None:
    raise RuntimeError("VideoWriter failed to open with codecs: " + ",".join(codecs))

frames_written = 0

def overlay_frame(row):
    # Decode image and actions
    img_chw = arr_from_bytes(row["observations.images.front_img_1"], IMG_SHAPE, np.uint8)
    img = np.transpose(img_chw, (1, 2, 0)).copy()  # HWC, RGB
    ee_chunk = arr_from_bytes(row["actions_ee_cartesian_cam"], ACTION_CHUNK, np.float32)
    right_xyz_cam = ee_chunk[:, 6:9]  # (100, 3)

    # Project 3D points to pixels and clamp to image bounds
    pixels = project(right_xyz_cam)
    pixels = np.nan_to_num(pixels, nan=0.0, posinf=0.0, neginf=0.0).astype(int)
    pixels[:, 0] = np.clip(pixels[:, 0], 0, width - 1)
    pixels[:, 1] = np.clip(pixels[:, 1], 0, height - 1)

    # Draw trajectory as a polyline and points
    for i in range(len(pixels) - 1):
        pt1 = (int(pixels[i, 0]), int(pixels[i, 1]))
        pt2 = (int(pixels[i + 1, 0]), int(pixels[i + 1, 1]))
        cv2.line(img, pt1, pt2, (0, 255, 0), 2)
    for pt in pixels[::10]:  # sparse markers along the path
        cv2.circle(img, (int(pt[0]), int(pt[1])), 3, (0, 0, 255), -1)

    # Convert RGB->BGR for OpenCV writer (expects contiguous uint8)
    return np.ascontiguousarray(cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

for _, row in df.iloc[:max_frames].iterrows():
    frame_bgr = overlay_frame(row)
    writer.write(frame_bgr)
    frames_written += 1

writer.release()
size_bytes = output_path.stat().st_size if output_path.exists() else 0
print(f"Codec used: {chosen}; wrote {frames_written} frames to {output_path} at {fps} fps; resolution {width}x{height}; size {size_bytes/1024:.1f} KB")

# Quick readability check: try to reopen and grab the first frame
cap = cv2.VideoCapture(str(output_path))
opened = cap.isOpened()
ret, frame = cap.read()
cap.release()
print(f"Readable by OpenCV: {opened and ret}")


In [ ]:
from scipy.spatial.transform import Rotation as R

# 1. Get Camera Pose for the current frame (image timestamp)
# This is needed to transform "World" frame keypoints into the "Camera" frame for projection
head_row = arr_from_bytes(row["actions_head_cartesian_world"], (10,), np.float32)
cam_pos = head_row[:3]
cam_quat = head_row[6:10]  # x, y, z, w

# Construct World-to-Camera transform
rot = R.from_quat(cam_quat)
T_c2w = np.eye(4)
T_c2w[:3, :3] = rot.as_matrix()
T_c2w[:3, 3] = cam_pos
T_w2c = np.linalg.inv(T_c2w)

# 2. Process all keypoints (100 steps, 126 floats -> left(21x3) + right(21x3))
# Reshape to (100 steps, 42 points, 3 coords)
all_kps_world = kp_chunk.reshape(100, 42, 3)

# Flatten to (N, 3) to transform in bulk
N = 100 * 42
flat_kps = all_kps_world.reshape(N, 3)
kps_homog = np.hstack([flat_kps, np.ones((N, 1))])

# Transform: P_cam = T_w2c @ P_world
kps_cam = (T_w2c @ kps_homog.T).T[:, :3]

# 3. Project to pixels
kps_pix = project(kps_cam)

# 4. Visualization
plt.figure(figsize=(10, 6))
plt.imshow(img_hwc)

# Reshape back to organize by hand and time
kps_pix_reshaped = kps_pix.reshape(100, 42, 2)
left_hand_pix = kps_pix_reshaped[:, :21, :].reshape(-1, 2)
right_hand_pix = kps_pix_reshaped[:, 21:, :].reshape(-1, 2)

# Create colors fading from dark to bright over the 100 steps
steps = np.arange(100)
colors = plt.cm.cool(steps / 100.0)  # (100, 4)
colors_repeated = np.repeat(colors, 21, axis=0)  # Repeat for each keypoint in a step

plt.scatter(left_hand_pix[:, 0], left_hand_pix[:, 1], c=colors_repeated, s=15, marker='o', label='Left Hand (World)', alpha=0.6, edgecolors='none')
plt.scatter(right_hand_pix[:, 0], right_hand_pix[:, 1], c=colors_repeated, s=15, marker='^', label='Right Hand (World)', alpha=0.6, edgecolors='none')

plt.title("Projected World Keypoints (100 steps)")
plt.legend(loc='upper right')
plt.axis("off")
plt.tight_layout()
plt.show()